In [ ]:
import os
import random
import itertools
import json
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import torchvision
from torchvision import transforms, models
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Device setup 
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
    
print(f'Using device: {device}')


In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)         
    torch.backends.cudnn.deterministic = True

seed_everything(7)

In [ ]:
# IMAGE TRANSFORMS 
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),                  
    transforms.ColorJitter(brightness=0.2, contrast=0.2),       
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

print('Transforms ready.')


In [ ]:
# DATA LOADING 
BASE_PATH = os.path.expanduser('~/Desktop/george/rocks_spectral_224')

folders_183 = {
    'S10Granite':           os.path.join(BASE_PATH, 'S10Granite_1-83Hz_Spectral'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'Holstein_Sandstone_1-83Hz_Spectral'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'Leitendorf_Limestone_1-83Hz_Spectral'),
}

folders_510 = {
    'S10Granite':           os.path.join(BASE_PATH, 'S10Granite_5-10Hz_Spectral'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'Holstein_Sandstone_5-10Hz_Spectral'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'Leitendorf_Limestone_5-10Hz_Spectral'),
}

VALID_EXTENSIONS = ('.jpg', '.jpeg', '.bmp', '.png')

def collect_image_paths(folders_dict):
    image_paths = []
    labels      = []
    class_names = list(folders_dict.keys())

    for label, (rock_name, folder) in enumerate(folders_dict.items()):
        files = [f for f in os.listdir(folder) if f.lower().endswith(VALID_EXTENSIONS)]
        for fname in files:
            image_paths.append(os.path.join(folder, fname))
            labels.append(label)
        print(f'{rock_name}: {len(files)} images')

    return image_paths, labels, class_names


print('Loading 1.83 Hz image paths')
paths_183, labels_183, class_names = collect_image_paths(folders_183)
print(f'Total: {len(paths_183)} images\n')

print('Loading 5.10 Hz image paths')
paths_510, labels_510, _ = collect_image_paths(folders_510)
print(f'Total: {len(paths_510)} images')


In [ ]:
class_names

In [ ]:
# VISUALIZE SAMPLE IMAGES 
for speed_tag, paths, labels in [
    ('1.83 Hz', paths_183, labels_183),
    ('5.10 Hz', paths_510, labels_510)
]:
    fig, axes = plt.subplots(3, 3, figsize=(12, 10))

    for i, rock_name in enumerate(class_names):
        rock_indices = [j for j, l in enumerate(labels) if l == i][:3]
        for k, idx in enumerate(rock_indices):
            img = Image.open(paths[idx]).convert('RGB')
            axes[i][k].imshow(img)
            axes[i][k].set_title(f'{rock_name}' if k == 1 else '')
            axes[i][k].axis('off')

    plt.suptitle(f'Sample Spectral Images - {speed_tag}', fontsize=14)
    plt.tight_layout()
    _tag6 = speed_tag.replace(' ', '').replace('.', '')
    import os as _os; _os.makedirs('results_resnet18_baseline', exist_ok=True)
    _sp6 = _os.path.join('results_resnet18_baseline', f'sample_images_{_tag6}.png')
    plt.savefig(_sp6, dpi=150, bbox_inches='tight')
    print(f'[SAVED] Sample images -> {_sp6}')
    plt.show()

In [ ]:
# TRAIN/TEST SPLIT 
(
    paths_train_183, paths_test_183,
    labels_train_183, labels_test_183
) = train_test_split(
    paths_183, labels_183,
    test_size=0.2, random_state=3, stratify=labels_183
)

(
    paths_train_510, paths_test_510,
    labels_train_510, labels_test_510
) = train_test_split(
    paths_510, labels_510,
    test_size=0.2, random_state=3, stratify=labels_510
)

print('1.83 Hz -> Train:', len(paths_train_183), ' Test:', len(paths_test_183))
print('5.10 Hz -> Train:', len(paths_train_510), ' Test:', len(paths_test_510))

In [ ]:
# DATASET CLASS 
class SpectralImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img   = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


train_dataset_183 = SpectralImageDataset(paths_train_183, labels_train_183, transform=train_transform)
test_dataset_183  = SpectralImageDataset(paths_test_183,  labels_test_183,  transform=test_transform)

train_dataset_510 = SpectralImageDataset(paths_train_510, labels_train_510, transform=train_transform)
test_dataset_510  = SpectralImageDataset(paths_test_510,  labels_test_510,  transform=test_transform)

print('Datasets ready.')

In [ ]:
# HYPERPARAMETERS 
batch_sizes   = [64]
lrs           = [0.0001, 0.00005]
epochs_list   = [20]
weight_decays = [0., 1e-4]

hyperparameters_list = []
for batch_size, lr, epochs, weight_decay in itertools.product(batch_sizes, lrs, epochs_list, weight_decays):
    hyperparameters_list.append({
        'batch_size'  : batch_size,
        'lr'          : lr,
        'epochs'      : epochs,
        'weight_decay': weight_decay
    })

print(f'Generated {len(hyperparameters_list)} combinations.')

In [ ]:
# RESNET-18 MODEL 

def build_resnet18(n_classes, freeze_backbone=False):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        # Freeze all layers except final classifier
        for param in model.parameters():
            param.requires_grad = False

    # Replace final fully connected layer
    in_features = model.fc.in_features  # 512
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, n_classes)
    )
    return model


# Test build
test_model = build_resnet18(n_classes=3)
total_params    = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
del test_model

In [ ]:
# TRAINING FUNCTION 
def run_experiments(train_dataset, test_dataset, speed_tag, results_dir, n_classes):

    experiment_results = []
    os.makedirs(results_dir, exist_ok=True)

    for idx, config in enumerate(hyperparameters_list):
        print(f"\n{'='*20}\nExperiment {idx+1}\nConfig: {config}\n{'='*20}")

        train_loader = DataLoader(
            train_dataset, batch_size=config['batch_size'],
            shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True
        )
        test_loader = DataLoader(
            test_dataset, batch_size=config['batch_size'],
            shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True
        )

        model     = build_resnet18(n_classes).to(device)
        optimizer = torch.optim.Adam(
            params=model.parameters(),
            lr=config['lr'],
            weight_decay=config['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=2
        )
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0, total_iters=2
        )
        criterion = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler(device.type if device.type == 'cuda' else 'cpu')

        best_accuracy   = 0.0
        best_model_path = os.path.join(results_dir, f'best_model_exp_{idx+1}.pth')
        writer          = SummaryWriter(log_dir=os.path.join(results_dir, f'tb_exp{idx+1}_lr{config["lr"]}_wd{config["weight_decay"]}'))

        training_loss, training_acc = [], []
        testing_loss,  testing_acc  = [], []

        for epoch in range(config['epochs']):
            print(f'Epoch: {epoch+1}/{config["epochs"]}')

            # Train
            model.train()
            avg_train_loss, avg_train_acc = [], []
            for X_batch, y_batch in tqdm(train_loader, leave=False):
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                optimizer.zero_grad()
                with torch.amp.autocast(device.type):
                    outputs = model(X_batch)
                    loss    = criterion(outputs, y_batch)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                scaler.step(optimizer)
                scaler.update()
                _, predicted = torch.max(outputs, 1)
                avg_train_loss.append(loss.item())
                avg_train_acc.append((predicted == y_batch).sum().item() / y_batch.size(0))

            avg_acc_train  = sum(avg_train_acc)  / len(avg_train_acc)
            avg_loss_train = sum(avg_train_loss) / len(avg_train_loss)
            training_acc.append(avg_acc_train)
            training_loss.append(avg_loss_train)
            print(f'Training: Loss={avg_loss_train:.4f}, Acc={avg_acc_train:.4f}')

            # Test
            model.eval()
            avg_test_loss, avg_test_acc = [], []
            all_preds, all_true = [], []
            with torch.no_grad():
                for X_batch, y_batch in tqdm(test_loader, leave=False):
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)
                    outputs = model(X_batch)
                    loss    = criterion(outputs, y_batch)
                    preds   = torch.max(outputs, 1)[1]
                    avg_test_loss.append(loss.item())
                    avg_test_acc.append((preds == y_batch).sum().item() / y_batch.size(0))
                    all_preds.extend(preds.cpu().numpy())
                    all_true.extend(y_batch.cpu().numpy())

            avg_acc_test  = sum(avg_test_acc)  / len(avg_test_acc)
            avg_loss_test = sum(avg_test_loss) / len(avg_test_loss)
            testing_acc.append(avg_acc_test)
            testing_loss.append(avg_loss_test)
            print(f'Testing:  Loss={avg_loss_test:.4f}, Acc={avg_acc_test:.4f}\n')

            if epoch < 2:
                warmup_scheduler.step()
            else:
                scheduler.step(avg_acc_test)

            writer.add_scalars('Loss',     {'train': avg_loss_train, 'test': avg_loss_test}, epoch)
            writer.add_scalars('Accuracy', {'train': avg_acc_train,  'test': avg_acc_test},  epoch)

            if avg_acc_test > best_accuracy:
                best_accuracy = avg_acc_test
                torch.save(model.state_dict(), best_model_path)
                print(f' New Best Acc: {best_accuracy:.4f} - Saved.')
            print('-' * 30)

        # Confusion Matrix using best checkpoint 
        model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
        model.to(device).eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for X_batch, y_batch in tqdm(test_loader, leave=False):
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                preds   = torch.max(model(X_batch), 1)[1]
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(y_batch.cpu().numpy())

        cm = confusion_matrix(all_true, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Confusion Matrix - {speed_tag} - Exp {idx+1}')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.tight_layout()
        cm_path = os.path.join(results_dir, f'confusion_matrix_exp_{idx+1}.png')
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'[SAVED] Confusion matrix -> {cm_path}')

        writer.close()
        del model, optimizer, scheduler, warmup_scheduler, train_loader, test_loader
        torch.cuda.empty_cache()  
        experiment_results.append({
            'config'          : config,
            'best_test_acc'   : best_accuracy,
            'final_test_acc'  : testing_acc[-1],
            'final_test_loss' : testing_loss[-1],
            'best_model_path' : best_model_path,
            'history': {
                'train_loss': training_loss, 'train_acc': training_acc,
                'test_loss' : testing_loss,  'test_acc' : testing_acc
            }
        })

    print('\n' + '='*40)
    print(f'RESULTS - {speed_tag}')
    print('='*40)
    for res in experiment_results:
        print(f"Config: {res['config']} -> Best Acc: {res['best_test_acc']:.4f}  Final Acc: {res['final_test_acc']:.4f}")

    return experiment_results


# RUN TRAINING 
print('\n' + '#'*50)
print('RESNET-18 TRAINING - 1.83 Hz')
print('#'*50)
results_183 = run_experiments(
    train_dataset_183, test_dataset_183,
    '1.83 Hz', 'results_resnet18_baseline/resnet18_results_183', n_classes=3
)

print('\n' + '#'*50)
print('RESNET-18 TRAINING - 5.10 Hz')
print('#'*50)
results_510 = run_experiments(
    train_dataset_510, test_dataset_510,
    '5.10 Hz', 'results_resnet18_baseline/resnet18_results_510', n_classes=3
)

In [ ]:
# ANALYZE ALL RESULTS
best_183 = max(results_183, key=lambda x: x['best_test_acc'])
best_510 = max(results_510, key=lambda x: x['best_test_acc'])

for speed_tag, results in [('1.83 Hz', results_183), ('5.10 Hz', results_510)]:
    print('=' * 70)
    print(f'RESNET-18 RESULTS — {speed_tag}')
    print('=' * 70)
    print(f"{'Exp':<5} {'Batch':<7} {'LR':<10} {'WD':<8} {'Best Acc':>10} {'Final Acc':>10} {'Final Loss':>11}")
    print('-' * 70)
    for i, res in enumerate(results):
        cfg = res['config']
        print(f"{i+1:<5} {cfg['batch_size']:<7} {cfg['lr']:<10} {cfg['weight_decay']:<8} "
              f"{res['best_test_acc']*100:>9.2f}% {res['final_test_acc']*100:>9.2f}% {res['final_test_loss']:>11.4f}")
    print()

# Plot ALL training curves
for speed_tag, results in [('1.83 Hz', results_183), ('5.10 Hz', results_510)]:
    n = len(results)
    fig, axes = plt.subplots(n, 2, figsize=(14, 4 * n), squeeze=False)
    fig.suptitle(f'ResNet-18 Training Curves — {speed_tag}', fontsize=14, fontweight='bold')

    for i, res in enumerate(results):
        cfg = res['config']
        label = f"Exp {i+1} | bs={cfg['batch_size']} lr={cfg['lr']} wd={cfg['weight_decay']}"

        axes[i][0].plot(res['history']['train_loss'], color='blue', label='Train')
        axes[i][0].plot(res['history']['test_loss'],  color='red',  label='Test')
        axes[i][0].set_title(f'Loss — {label}')
        axes[i][0].set_xlabel('Epoch')
        axes[i][0].set_ylabel('Loss')
        axes[i][0].legend()
        axes[i][0].grid(True, alpha=0.3)

        axes[i][1].plot(res['history']['train_acc'], color='blue', label='Train')
        axes[i][1].plot(res['history']['test_acc'],  color='red',  label='Test')
        axes[i][1].set_title(f'Accuracy — {label} | Best: {res["best_test_acc"]*100:.2f}%')
        axes[i][1].set_xlabel('Epoch')
        axes[i][1].set_ylabel('Accuracy')
        axes[i][1].legend()
        axes[i][1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.95)
    _dir12 = 'results_resnet18_baseline/resnet18_results_183' if '1.83' in speed_tag else 'results_resnet18_baseline/resnet18_results_510'
    import os as _os; _os.makedirs(_dir12, exist_ok=True)
    _cp12 = _os.path.join(_dir12, 'training_curves_all.png')
    plt.savefig(_cp12, dpi=150, bbox_inches='tight')
    print(f'[SAVED] Training curves -> {_cp12}')
    plt.show()

In [ ]:
# CONFUSION MATRICES - ALL EXPERIMENTS
for speed_tag, results, test_ds in [
    ('1.83Hz', results_183, test_dataset_183),
    ('5.10Hz', results_510, test_dataset_510)
]:
    for res in results:
        cfg = res['config']
        label = f"bs={cfg['batch_size']} lr={cfg['lr']} wd={cfg['weight_decay']}"

        y_true, y_pred = [], []
        loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
        model  = build_resnet18(n_classes=3).to(device)
        model.load_state_dict(torch.load(res['best_model_path'], map_location=device, weights_only=True))
        model.eval()

        with torch.no_grad():
            for X_batch, y_batch in loader:
                outputs      = model(X_batch.to(device))
                _, predicted = torch.max(outputs, 1)
                y_true.extend(y_batch.tolist())
                y_pred.extend(predicted.cpu().tolist())

        cm   = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
        fig, ax = plt.subplots(figsize=(8, 8))
        disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
        plt.title(f'ResNet-18 Confusion Matrix - {speed_tag} - {label}\nBest Acc: {res["best_test_acc"]*100:.2f}%')
        _rdir13 = os.path.dirname(res['best_model_path'])
        _cm13 = f'confusion_matrix_postrun_{speed_tag.replace(" ","").replace(".","")}_bs{cfg["batch_size"]}_lr{cfg["lr"]}_wd{cfg["weight_decay"]}.png'
        _cmp13 = os.path.join(_rdir13, _cm13)
        plt.savefig(_cmp13, dpi=150, bbox_inches='tight')
        print(f'[SAVED] Post-run CM -> {_cmp13}')
        plt.show()

        del model
        torch.cuda.empty_cache()
        

In [ ]:
#  SAVE BEST MODELS
import shutil 
for tag, best in [('1.83Hz', best_183), ('5.10Hz', best_510)]:
    os.makedirs('results_resnet18_baseline', exist_ok=True)
    dst = f'results_resnet18_baseline/rock_classifier_resnet18_{tag}.pth'
    shutil.copy(best['best_model_path'], dst)
    with open(f'results_resnet18_baseline/class_names_resnet18_{tag}.json', 'w') as f:
        json.dump(class_names, f)
    print(f'Saved {dst}')

print('\nTo view TensorBoard:')
print('  cd ~/results_resnet18_baseline/resnet18_results_183 && tensorboard --logdir=.')
print('  cd ~/results_resnet18_baseline/resnet18_results_510 && tensorboard --logdir=.')

summary = {
    '1.83Hz': {'best_acc': best_183['best_test_acc'], 'config': best_183['config']},
    '5.10Hz': {'best_acc': best_510['best_test_acc'], 'config': best_510['config']},
    'class_names': class_names
}
with open('results_resnet18_baseline/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)